# Clinical Decision Support System (CDSS)
### Patient Research Tool — Google Colab Edition

**What this notebook does:**
- Parses your condition description or uploaded test results
- Retrieves standard-of-care treatment options from medical guidelines
- Searches active/recruiting clinical trials worldwide (ClinicalTrials.gov)
- Discovers off-label therapy hypotheses via biomedical knowledge graph traversal
- Compiles a plain-English research report with questions to ask your doctor

**Model:** DeepSeek R1 via NVIDIA API *(free tier available at build.nvidia.com)*

> ⚠️ **This tool is for research and education only. It is not medical advice.**

---
### Colab-Specific Architecture

| Component | Local Version | Colab Version |
|-----------|--------------|---------------|
| Vector DB | Qdrant (Docker) | Qdrant in-memory |
| Knowledge Graph | Neo4j (Docker) | NetworkX in-memory |
| Secrets | `.env` file | Colab Secrets / getpass |
| PDF Input | Local file path | Colab file upload |

---
## Cell 1 — Install Dependencies

In [ ]:
!pip install -q \
    openai \
    langgraph \
    pydantic \
    qdrant-client \
    sentence-transformers \
    networkx \
    pypdf

print("✅ Dependencies installed.")

---
## Cell 2 — Configure API Key

**Recommended:** Add your key to Colab Secrets (🔑 icon in the left sidebar) as `GROQ_API_KEY`.  
If not set there, you will be prompted to enter it manually.

Get a free Groq API key at **https://console.groq.com/keys**

In [ ]:
import os
import getpass

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = getpass.getpass("Enter your Groq API key (gsk_...): ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("✅ API key configured.")

---
## Cell 3 — Imports & State Model

In [ ]:
import os
import json
import urllib.parse
import urllib.request
import csv
from pathlib import Path
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field
from openai import OpenAI


# ── State Model ────────────────────────────────────────────────────────────────

class Biomarker(BaseModel):
    gene: str
    variant_type: str
    details: str


class ClinicalTrial(BaseModel):
    nct_id: str
    title: str
    phase: str
    status: str
    locations: List[str]
    eligibility_summary: str
    url: str


class OffLabelHypothesis(BaseModel):
    drug_name: str
    approved_indication: str
    shared_mechanism: str
    evidence_level: int   # 1=in-vitro, 2=animal, 3=Phase I/II, 4=Phase III adjacent
    evidence_label: str
    citation: str


class PatientProfileState(BaseModel):
    raw_input: str
    input_is_pdf: bool = False
    condition: str = ""
    stage: str = ""
    biomarkers: List[Biomarker] = Field(default_factory=list)
    current_medications: List[str] = Field(default_factory=list)
    prior_therapies: List[str] = Field(default_factory=list)
    standard_therapies: List[Dict[str, Any]] = Field(default_factory=list)
    clinical_trials: List[ClinicalTrial] = Field(default_factory=list)
    off_label_hypotheses: List[OffLabelHypothesis] = Field(default_factory=list)
    validation_flags: List[str] = Field(default_factory=list)
    final_report: str = ""
    retry_count: int = 0
    max_retries: int = 2


print("✅ State model defined.")

---
## Cell 4 — LLM Client (Groq — DeepSeek R1)

Groq hosts DeepSeek R1 distill models with very low latency. No model discovery needed — the API is stable.

In [ ]:
from openai import OpenAI

llm = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)

# ── Fetch currently available models from Groq ────────────────────────────────
print("Fetching available Groq models...")
available_models = {m.id for m in llm.models.list().data}
print(f"  {len(available_models)} models available:")
for m in sorted(available_models):
    print(f"    - {m}")

# ── Pick best model: prefer large DeepSeek, fall back to large Llama ──────────
PREFERENCE_ORDER = [
    # DeepSeek R1 variants (largest first)
    "deepseek-r1-distill-llama-70b",
    "deepseek-r1-distill-qwen-32b",
    "deepseek-r1-distill-llama-8b",
    # Llama 3.x fallbacks
    "llama-3.3-70b-versatile",
    "llama-3.1-70b-versatile",
    "llama3-70b-8192",
    "llama-3.1-8b-instant",
    "llama3-8b-8192",
]

MODEL = next((m for m in PREFERENCE_ORDER if m in available_models), None)

if MODEL is None:
    # Last resort: pick whatever is available with the largest context
    MODEL = sorted(available_models)[0]
    print(f"  ⚠️  No preferred model found — using: {MODEL}")

print(f"\n✅ Selected model: {MODEL}")

# ── Manual override — uncomment to force a specific model ─────────────────────
# MODEL = "llama-3.3-70b-versatile"


# ── Shared helpers ─────────────────────────────────────────────────────────────
def chat(prompt: str, max_tokens: int = 1024) -> str:
    resp = llm.chat.completions.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content.strip()


def strip_json_fences(text: str) -> str:
    """Remove markdown ```json ... ``` fences that models sometimes add."""
    text = text.strip()
    if text.startswith("```"):
        parts = text.split("```")
        text = parts[1]
        if text.startswith("json"):
            text = text[4:]
    return text.strip()


# Sanity check
print(chat("Reply with exactly three words: Groq API OK", max_tokens=20))

---
## Cell 5 — Vector Store (Qdrant In-Memory)

No Docker needed — Qdrant runs entirely in RAM for this session.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

COLLECTION = "guidelines"
EMBED_MODEL = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
VECTOR_DIM  = 768

print("Loading biomedical embedding model (downloads once, ~400 MB)...")
embedder = SentenceTransformer(EMBED_MODEL)

qdrant = QdrantClient(":memory:")
qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
)


def qdrant_search(query: str, top_k: int = 5) -> List[dict]:
    vec = embedder.encode([query])[0].tolist()
    # query_points() replaced search() in qdrant-client >= 1.7
    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=vec,
        limit=top_k,
    ).points
    return [{"text": h.payload.get("text", ""), "source": h.payload.get("source", ""), "score": h.score} for h in results]


def chunk_text(text: str, size: int = 512, overlap: int = 64) -> List[str]:
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size-overlap) if words[i:i+size]]


print("✅ Vector store ready (in-memory Qdrant).")

---
## Cell 6 — Upload Clinical Guidelines (Optional)

Upload NCCN, ESMO, or any guideline PDFs. The more you upload, the richer the standard-care recommendations.  
**This cell is optional** — if you skip it, Agent 2 will note that no guideline data is available.

In [ ]:
import pypdf
from google.colab import files as colab_files

print("Select one or more guideline PDF files to upload (or skip this cell).")
uploaded = colab_files.upload()   # opens file picker

point_id = 0
for filename, content in uploaded.items():
    # Write bytes to a temp path so pypdf can read it
    tmp_path = f"/tmp/{filename}"
    with open(tmp_path, "wb") as f:
        f.write(content)

    reader = pypdf.PdfReader(tmp_path)
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    chunks = chunk_text(text)
    vectors = embedder.encode(chunks, show_progress_bar=True).tolist()

    points = [
        PointStruct(id=point_id + i, vector=v, payload={"source": filename, "text": chunks[i]})
        for i, v in enumerate(vectors)
    ]
    qdrant.upsert(collection_name=COLLECTION, points=points)
    point_id += len(points)
    print(f"  ✅ Ingested {len(chunks)} chunks from {filename}")

print(f"\nTotal chunks in vector store: {point_id}")

---
## Cell 7 — Knowledge Graph (NetworkX + PrimeKG)

Loads PrimeKG into an in-memory NetworkX graph. Downloads from Harvard Dataverse (~200 MB).  
**This cell is optional** — skip it and Agent 4 will note that the knowledge graph is unavailable.

> ⏱️ First run takes ~5 minutes to download + load. Subsequent runs in the same session are instant.

In [ ]:
import networkx as nx
import requests
import csv
import os

PRIMEKG_DOI   = "doi:10.7910/DVN/IXA7BM"
DATAVERSE_API = "https://dataverse.harvard.edu/api"
HEADERS       = {"User-Agent": "Mozilla/5.0 (compatible; PrimeKG-loader/1.0)"}

KG = nx.MultiDiGraph()
KG_AVAILABLE = False


def get_dataset_files() -> list:
    """Return the full file manifest from Harvard Dataverse."""
    url  = f"{DATAVERSE_API}/datasets/:persistentId/?persistentId={PRIMEKG_DOI}"
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json()["data"]["latestVersion"]["files"]


def dataverse_download(file_id: int, dest: str):
    url = f"{DATAVERSE_API}/access/datafile/{file_id}"
    with requests.get(url, headers=HEADERS, stream=True, timeout=300) as r:
        r.raise_for_status()
        total      = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"\r    {downloaded/1e6:.1f} / {total/1e6:.1f} MB", end="")
    print()


def pick_file(files: list, keywords: list) -> tuple:
    """Return (filename, file_id) for the first manifest entry whose name contains any keyword."""
    for entry in files:
        name = entry["dataFile"]["filename"].lower()
        if any(kw in name for kw in keywords):
            return entry["dataFile"]["filename"], entry["dataFile"]["id"]
    return None, None


try:
    # ── Step 1: list every file in the dataset ────────────────────────────────
    print("Querying Harvard Dataverse manifest...")
    all_files = get_dataset_files()
    print(f"  Files in dataset ({len(all_files)} total):")
    for f in all_files:
        df = f["dataFile"]
        size_mb = df.get("filesize", 0) / 1e6
        print(f"    [{df['id']}]  {df['filename']}  ({size_mb:.1f} MB)")

    # ── Step 2: flexible filename matching ────────────────────────────────────
    nodes_name, nodes_id = pick_file(all_files, ["node"])
    edges_name, edges_id = pick_file(all_files, ["edge", "kg_raw", "kg.csv", "relations"])

    print(f"\n  → Nodes file : {nodes_name}  (id={nodes_id})")
    print(f"  → Edges file : {edges_name}  (id={edges_id})")

    if not nodes_id or not edges_id:
        raise RuntimeError(
            "Could not auto-detect nodes/edges files from the manifest above.\n"
            "Set nodes_id and edges_id manually using the IDs printed above."
        )

    # ── Manual override — uncomment and set IDs from the manifest above ───────
    # nodes_id, nodes_name = 1234567, "nodes.csv"
    # edges_id, edges_name = 1234568, "edges.csv"

    # ── Step 3: download ──────────────────────────────────────────────────────
    nodes_path = "/tmp/primekg_nodes.csv"
    edges_path = "/tmp/primekg_edges.csv"

    for fid, fname, dest in [(nodes_id, nodes_name, nodes_path),
                              (edges_id, edges_name, edges_path)]:
        if os.path.exists(dest):
            print(f"  {fname} already cached — skipping.")
        else:
            print(f"  Downloading {fname}...")
            dataverse_download(fid, dest)
            print(f"  ✅ Saved to {dest}")

    # ── Step 4: load nodes ────────────────────────────────────────────────────
    print("\n  Loading nodes...")
    with open(nodes_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        cols = reader.fieldnames
        print(f"  Columns: {cols}")
        # Flexible column mapping
        id_col   = next((c for c in cols if "index" in c or c == "id"), cols[0])
        name_col = next((c for c in cols if "name"  in c), cols[1] if len(cols) > 1 else cols[0])
        type_col = next((c for c in cols if "type"  in c), None)
        for row in reader:
            nid  = row.get(id_col, "")
            name = row.get(name_col, "")
            ntype = row.get(type_col, "") if type_col else ""
            KG.add_node(nid, name=name, type=ntype)
    print(f"  Nodes loaded: {KG.number_of_nodes():,}")

    # ── Step 5: load edges ────────────────────────────────────────────────────
    print("  Loading edges (may take ~2 min)...")
    with open(edges_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        cols = reader.fieldnames
        print(f"  Columns: {cols}")
        src_col = next((c for c in cols if c in ("x_index", "head_index", "source", "src")), cols[0])
        dst_col = next((c for c in cols if c in ("y_index", "tail_index", "target", "dst")), cols[1])
        rel_col = next((c for c in cols if "relation" in c or "type" in c), cols[2] if len(cols) > 2 else None)
        for row in reader:
            src = row.get(src_col, "")
            dst = row.get(dst_col, "")
            rel = row.get(rel_col, "RELATED_TO") if rel_col else "RELATED_TO"
            KG.add_edge(src, dst, relation=rel)
    print(f"  Edges loaded: {KG.number_of_edges():,}")

    KG_AVAILABLE = True
    print("\n✅ PrimeKG loaded into NetworkX.")

except Exception as e:
    print(f"\n⚠️  PrimeKG unavailable: {e}")
    print("   Agent 4 (cross-indication) will be skipped. All other agents still run.")

---
## Cell 8 — Knowledge Graph Query Functions

In [ ]:
def find_node_by_name(name: str) -> Optional[str]:
    """Return node_id for a gene/drug name (case-insensitive partial match)."""
    name_lower = name.lower()
    for nid, attrs in KG.nodes(data=True):
        if name_lower in attrs.get("name", "").lower():
            return nid
    return None


def find_drugs_for_gene(gene: str, max_hops: int = 2, limit: int = 15) -> List[dict]:
    """
    Traverse: Gene → (pathway/protein) → Drug (approved for another disease).
    Returns list of {drug, approved_indication, shared_pathway}.
    """
    if not KG_AVAILABLE:
        return []

    gene_node = find_node_by_name(gene)
    if not gene_node:
        return []

    # BFS from gene node up to max_hops
    visited = {gene_node}
    frontier = {gene_node}
    for _ in range(max_hops):
        next_frontier = set()
        for n in frontier:
            next_frontier.update(KG.successors(n))
            next_frontier.update(KG.predecessors(n))
        frontier = next_frontier - visited
        visited.update(frontier)

    results = []
    seen_drugs = set()
    for nid in visited:
        attrs = KG.nodes[nid]
        if attrs.get("type", "").lower() in ("drug", "small molecule", "compound"):
            drug_name = attrs.get("name", "")
            if drug_name in seen_drugs:
                continue
            seen_drugs.add(drug_name)

            # Find diseases this drug is connected to
            indications = []
            for neighbor in list(KG.successors(nid)) + list(KG.predecessors(nid)):
                n_attrs = KG.nodes[neighbor]
                if n_attrs.get("type", "").lower() in ("disease", "phenotype", "indication"):
                    indications.append(n_attrs.get("name", ""))

            results.append({
                "drug": drug_name,
                "approved_indication": ", ".join(indications[:2]) if indications else "see literature",
                "shared_pathway": f"connected via {max_hops}-hop path from {gene}",
            })
            if len(results) >= limit:
                break

    return results


print("✅ Knowledge graph query functions defined.")

---
## Cell 9 — Agent 1: Intake & Parse

In [ ]:
INTAKE_PROMPT = """You are a medical information extractor. A patient has provided information about their condition.
Extract the following fields as JSON. Only extract what is clearly stated — do not infer or guess.

Patient input:
{raw_input}

Rules for biomarkers:
- If a fusion gene is mentioned (e.g. "DNAJB1-PRKACA fusion"), create TWO entries:
  one for the full fusion name and one for each component gene separately.
- variant_type options: Mutation, Fusion, Amplification, Deletion, Overexpression

Return ONLY valid JSON with this exact structure (use empty strings / empty arrays if not present):
{{
  "condition": "primary diagnosis or disease name",
  "stage": "disease stage or severity if mentioned (e.g. Stage IV, Metastatic)",
  "biomarkers": [
    {{"gene": "GENE_NAME", "variant_type": "Mutation|Fusion|Amplification|Deletion|Overexpression", "details": "specific description"}}
  ],
  "current_medications": ["medication name"],
  "prior_therapies": ["therapy name"]
}}"""


def _expand_fusions(biomarkers: list) -> list:
    """
    For any fusion biomarker (e.g. DNAJB1-PRKACA), add the component genes
    as individual entries so graph traversal can find each one separately.
    """
    expanded = list(biomarkers)
    for bm in biomarkers:
        if bm.variant_type == "Fusion" and "-" in bm.gene:
            parts = bm.gene.split("-")
            for part in parts:
                expanded.append(Biomarker(
                    gene=part,
                    variant_type="Fusion component",
                    details=f"Component of {bm.gene} fusion",
                ))
    return expanded


def agent_intake(state: PatientProfileState) -> PatientProfileState:
    raw = state.raw_input

    if state.input_is_pdf:
        try:
            reader = pypdf.PdfReader(raw)
            raw = "\n".join(page.extract_text() or "" for page in reader.pages)
        except Exception as e:
            state.validation_flags.append(f"PDF extraction error: {e}")
            return state

    text = strip_json_fences(chat(INTAKE_PROMPT.format(raw_input=raw)))

    try:
        data = json.loads(text)
        state.condition           = data.get("condition", "")
        state.stage               = data.get("stage", "")
        state.current_medications = data.get("current_medications", [])
        state.prior_therapies     = data.get("prior_therapies", [])
        raw_biomarkers = [
            Biomarker(gene=b.get("gene",""), variant_type=b.get("variant_type",""), details=b.get("details",""))
            for b in data.get("biomarkers", [])
        ]
        # Expand fusions into component genes for better KG traversal
        state.biomarkers = _expand_fusions(raw_biomarkers)
    except (json.JSONDecodeError, KeyError) as e:
        state.validation_flags.append(f"Intake parse error: {e}")

    return state


print("✅ Agent 1 defined.")

---
## Cell 10 — Agent 2: Standard Care Retriever

In [ ]:
CARE_PROMPT = """You are helping a patient understand standard treatment options for their condition.
Based ONLY on the following excerpts from medical guidelines, summarise the standard of care in plain English.
Do not add information not present in the excerpts. Do not mention specific doses.
End with: "Ask your doctor which of these options apply to your specific situation."

Patient condition: {condition} — Stage: {stage}
Prior therapies already tried: {prior_therapies}

Guideline excerpts:
{excerpts}

Write a plain-English summary (3-5 sentences per treatment category)."""


def agent_standard_care(state: PatientProfileState) -> PatientProfileState:
    if not state.condition:
        state.validation_flags.append("Standard care skipped: no condition extracted.")
        return state

    hits = qdrant_search(f"{state.condition} {state.stage} treatment guidelines", top_k=6)

    if not hits:
        state.standard_therapies = [{"summary": (
            "No guideline data was found in the knowledge base. "
            "Upload NCCN or ESMO guideline PDFs in Cell 6 and re-run, "
            "or ask your doctor directly about standard-of-care options for your condition."
        )}]
        return state

    excerpts = "\n\n---\n\n".join(h["text"] for h in hits)
    summary = chat(CARE_PROMPT.format(
        condition=state.condition,
        stage=state.stage or "unspecified",
        prior_therapies=", ".join(state.prior_therapies) or "none mentioned",
        excerpts=excerpts,
    ))

    state.standard_therapies = [{
        "summary": summary,
        "sources": list({h["source"] for h in hits}),
    }]
    return state


print("✅ Agent 2 defined.")

---
## Cell 11 — Agent 3: Clinical Trials Finder

In [ ]:
TRIALS_BASE = "https://clinicaltrials.gov/api/v2/studies"


def fetch_trials(condition: str, biomarker_terms: List[str], max_results: int = 10) -> List[ClinicalTrial]:
    params = {
        "query.cond": condition,
        "filter.overallStatus": "RECRUITING",
        "pageSize": max_results,
        "format": "json",
        "fields": "NCTId,BriefTitle,Phase,OverallStatus,LocationCity,LocationCountry,EligibilityCriteria",
    }
    if biomarker_terms:
        params["query.term"] = " OR ".join(biomarker_terms)

    url = TRIALS_BASE + "?" + urllib.parse.urlencode({k: v for k, v in params.items() if v})
    try:
        with urllib.request.urlopen(url, timeout=15) as resp:
            data = json.loads(resp.read().decode())
    except Exception as e:
        return [], str(e)

    results = []
    for study in data.get("studies", []):
        proto  = study.get("protocolSection", {})
        id_mod = proto.get("identificationModule", {})
        st_mod = proto.get("statusModule", {})
        de_mod = proto.get("designModule", {})
        lc_mod = proto.get("contactsLocationsModule", {})
        el_mod = proto.get("eligibilityModule", {})

        nct_id = id_mod.get("nctId", "")
        phases = de_mod.get("phases", [])
        locations = list({
            f"{loc.get('city','')}, {loc.get('country','')}".strip(", ")
            for loc in lc_mod.get("locations", []) if loc.get("city") or loc.get("country")
        })[:5]
        eligibility = el_mod.get("eligibilityCriteria", "")

        results.append(ClinicalTrial(
            nct_id=nct_id,
            title=id_mod.get("briefTitle", ""),
            phase=", ".join(phases) if phases else "N/A",
            status=st_mod.get("overallStatus", ""),
            locations=locations,
            eligibility_summary=(eligibility[:500] + "...") if len(eligibility) > 500 else eligibility,
            url=f"https://clinicaltrials.gov/study/{nct_id}",
        ))

    return results, None


def agent_clinical_trials(state: PatientProfileState) -> PatientProfileState:
    if not state.condition:
        state.validation_flags.append("Clinical trials skipped: no condition extracted.")
        return state

    biomarker_terms = [b.gene for b in state.biomarkers if b.gene]
    trials, error = fetch_trials(state.condition, biomarker_terms)

    if error:
        state.validation_flags.append(f"Trials API error: {error}")
    else:
        state.clinical_trials = trials

    return state


print("✅ Agent 3 defined.")

---
## Cell 12 — Agent 4: Cross-Indication Discoverer

In [ ]:
EVIDENCE_LABELS = {
    1: "Pre-clinical (in-vitro only)",
    2: "Pre-clinical (animal models)",
    3: "Early clinical (Phase I/II data)",
    4: "Clinical (Phase III data for adjacent indication)",
}

# ── Prompt A: KG candidates available ────────────────────────────────────────
HYPOTHESIS_PROMPT_KG = """You are an expert oncologist and molecular biologist.

A patient has {condition} (stage: {stage}).
Genomic profile: {biomarkers}
Prior treatments that failed: {prior_therapies}

The following drugs were found via biological pathway analysis to be potentially relevant.
They are approved for OTHER conditions but share a mechanism with this patient's alterations:
{candidates}

For each drug, explain:
1. The precise biological chain linking the patient's genomic alteration to this drug's target
2. Honest evidence level: 1=in-vitro only, 2=animal models, 3=Phase I/II human data, 4=Phase III adjacent
3. A PubMed search URL

Return ONLY a JSON array:
[
  {{
    "drug_name": "generic drug name",
    "approved_indication": "what it is approved for",
    "shared_mechanism": "2-3 sentence plain English biological rationale",
    "evidence_level": 1|2|3|4,
    "citation": "https://pubmed.ncbi.nlm.nih.gov/?term=DRUG+GENE"
  }}
]
Only include drugs with a clear mechanistic rationale. Return [] if none apply."""


# ── Prompt B: LLM-only fallback ───────────────────────────────────────────────
HYPOTHESIS_PROMPT_LLM = """You are an expert oncologist and molecular biologist specializing in rare and refractory cancers.

A patient has {condition} (stage: {stage}).
Genomic profile: {biomarkers}
Prior treatments that failed: {prior_therapies}

Your task: identify up to 8 off-label drug candidates approved for OTHER conditions that have a
strong mechanistic rationale for this patient's specific genomic alterations.

Reasoning framework — apply ALL of the following lenses to every alteration listed:

1. LOSS-OF-FUNCTION / DELETION alterations (e.g. SMARCB1 loss, PTEN loss, RB1 loss):
   - Ask: what pathway is now UNRESTRAINED because this suppressor is gone?
   - Ask: what is the synthetic lethal partner of this lost gene?
   - Examples: SMARCB1 loss → unopposed EZH2 activity → EZH2 inhibitors (Tazemetostat)
               PTEN loss → PI3K/AKT/mTOR hyperactivation → mTOR inhibitors (Everolimus, Temsirolimus)
               BRCA loss → HRD → PARP inhibitors (Olaparib)

2. GAIN-OF-FUNCTION / FUSION / AMPLIFICATION alterations:
   - Ask: what kinase or pathway is constitutively activated?
   - Ask: what approved inhibitor targets that kinase or downstream node?

3. CHROMATIN REMODELING complex loss (SWI/SNF: SMARCB1, SMARCA4, ARID1A etc.):
   - Always consider: EZH2 inhibitors, HDAC inhibitors, BET bromodomain inhibitors

4. TISSUE-AGNOSTIC approvals:
   - Check if any FDA-approved drug targets a biomarker REGARDLESS of tumor type
   - Examples: Tazemetostat for SMARCB1-loss tumors, Larotrectinib for NTRK fusions,
     Pembrolizumab for TMB-High or MSI-H, Dabrafenib/Trametinib for BRAF V600E

5. BASKET TRIAL / CASE REPORT evidence:
   - Even if evidence is pre-clinical, flag drugs that have shown activity in cell lines
     or animal models of tumors with this exact alteration

For each candidate, be specific: name the exact molecular chain (e.g. "SMARCB1 loss removes
BAF complex repression of EZH2, leading to H3K27me3 hypermethylation and silencing of tumor
suppressors. Tazemetostat inhibits EZH2 and restores tumor suppressor expression.").

Return ONLY a JSON array (no markdown, no prose outside the array):
[
  {{
    "drug_name": "generic name",
    "approved_indication": "current approved use",
    "shared_mechanism": "specific 2-3 sentence mechanistic rationale for THIS patient's alteration",
    "evidence_level": 1|2|3|4,
    "citation": "https://pubmed.ncbi.nlm.nih.gov/?term=DRUG+ALTERATION+cancer"
  }}
]

Evidence: 1=in-vitro, 2=animal model, 3=Phase I/II human data, 4=Phase III data for adjacent indication.
Do NOT return an empty array unless you are certain no mechanistic rationale exists."""


def agent_cross_indication(state: PatientProfileState) -> PatientProfileState:
    if not state.biomarkers:
        state.validation_flags.append("Cross-indication skipped: no biomarkers extracted.")
        return state

    biomarkers_text = ", ".join(
        f"{b.gene} {b.variant_type} ({b.details})" for b in state.biomarkers
    )
    prior_text = ", ".join(state.prior_therapies) or "none"

    # ── KG traversal (3 hops) ─────────────────────────────────────────────────
    all_candidates, seen = [], set()
    if KG_AVAILABLE:
        for bm in state.biomarkers:
            for r in find_drugs_for_gene(bm.gene, max_hops=3):
                key = r.get("drug", "")
                if key and key not in seen:
                    seen.add(key)
                    all_candidates.append(r)

    # ── Route: enough KG hits → grounded prompt ───────────────────────────────
    if len(all_candidates) >= 3:
        candidates_text = "\n".join(
            f"- {c['drug']} (approved for: {c['approved_indication']}, via: {c['shared_pathway']})"
            for c in all_candidates[:15]
        )
        prompt = HYPOTHESIS_PROMPT_KG.format(
            condition=state.condition,
            stage=state.stage or "unspecified",
            biomarkers=biomarkers_text,
            prior_therapies=prior_text,
            candidates=candidates_text,
        )
        source = "knowledge graph + LLM"

    # ── Fallback: sparse KG → pure LLM biomedical reasoning ──────────────────
    else:
        if KG_AVAILABLE:
            state.validation_flags.append(
                f"KG returned only {len(all_candidates)} candidates "
                f"(alteration may be rare or absent in PrimeKG). Using LLM reasoning fallback."
            )
        else:
            state.validation_flags.append(
                "PrimeKG not loaded — using LLM biomedical reasoning for cross-indication discovery."
            )
        prompt = HYPOTHESIS_PROMPT_LLM.format(
            condition=state.condition,
            stage=state.stage or "unspecified",
            biomarkers=biomarkers_text,
            prior_therapies=prior_text,
        )
        source = "LLM biomedical reasoning"

    print(f"  Cross-indication source: {source}")
    text = strip_json_fences(chat(prompt, max_tokens=3000))

    try:
        for h in json.loads(text):
            level = int(h.get("evidence_level", 1))
            state.off_label_hypotheses.append(OffLabelHypothesis(
                drug_name=h.get("drug_name", ""),
                approved_indication=h.get("approved_indication", ""),
                shared_mechanism=h.get("shared_mechanism", ""),
                evidence_level=level,
                evidence_label=EVIDENCE_LABELS.get(level, "Unknown"),
                citation=h.get("citation", ""),
            ))
    except (json.JSONDecodeError, KeyError) as e:
        state.validation_flags.append(f"Cross-indication parse error: {e}")

    return state


print("✅ Agent 4 defined.")

---
## Cell 13 — Agent 5: Report Synthesizer

In [ ]:
DISCLAIMER = """
---
**⚠️ Important Disclaimer**

This report was generated by an AI research tool and is for **informational and educational purposes only**.
It does **not** constitute medical advice, diagnosis, or a treatment recommendation.

- Clinical trial eligibility must be confirmed by a qualified physician
- Off-label therapy hypotheses require evaluation by your specialist before any consideration
- Never start, stop, or change a treatment based on this report alone

Always consult a licensed medical professional before making any treatment decisions.
"""


def validate_nct_id(nct_id: str) -> bool:
    try:
        url = f"https://clinicaltrials.gov/api/v2/studies/{nct_id}?format=json&fields=NCTId"
        with urllib.request.urlopen(url, timeout=5) as resp:
            return resp.status == 200
    except Exception:
        return False


def agent_synthesizer(state: PatientProfileState) -> PatientProfileState:
    lines = []

    lines += ["# Patient Research Report",
              "*Generated by CDSS — for informational purposes only. Not medical advice.*\n"]

    # ── Profile ──────────────────────────────────────────────────────────────
    lines.append("## Your Profile\n")
    lines.append(f"- **Condition:** {state.condition or 'Not specified'}")
    lines.append(f"- **Stage / Severity:** {state.stage or 'Not specified'}")
    if state.biomarkers:
        bm_str = ", ".join(f"{b.gene} {b.variant_type} ({b.details})" for b in state.biomarkers)
        lines.append(f"- **Key Biomarkers:** {bm_str}")
    if state.current_medications:
        lines.append(f"- **Current Medications:** {', '.join(state.current_medications)}")
    if state.prior_therapies:
        lines.append(f"- **Prior Therapies:** {', '.join(state.prior_therapies)}")
    lines.append("")

    # ── Standard Care ────────────────────────────────────────────────────────
    lines.append("## Standard Treatment Options\n")
    lines.append(state.standard_therapies[0]["summary"] if state.standard_therapies
                 else "*No guideline data available. Upload guideline PDFs in Cell 6.*")
    lines.append("")

    # ── Clinical Trials ──────────────────────────────────────────────────────
    lines.append("## Active Clinical Trials (Worldwide)\n")
    print("  Validating NCT IDs...")
    valid_trials = [t for t in state.clinical_trials if validate_nct_id(t.nct_id)]
    for t in state.clinical_trials:
        if t not in valid_trials:
            state.validation_flags.append(f"Trial {t.nct_id} could not be verified — excluded.")

    if valid_trials:
        lines += ["| Trial | Phase | Locations | NCT ID |",
                  "|-------|-------|-----------|--------|"]
        for t in valid_trials:
            locs  = "; ".join(t.locations[:3]) or "See link"
            title = (t.title[:55] + "...") if len(t.title) > 55 else t.title
            lines.append(f"| [{title}]({t.url}) | {t.phase} | {locs} | {t.nct_id} |")
        lines += ["", "> **Next step:** Ask your doctor: *\"Do I qualify for [trial name] given my profile?\"*"]
    else:
        lines.append("*No recruiting trials found matching your profile at this time.*")
    lines.append("")

    # ── Off-Label Hypotheses ─────────────────────────────────────────────────
    lines.append("## Off-Label Therapy Hypotheses\n")
    lines.append("Treatments approved for **other conditions** that share biological pathways with yours. "
                 "Require physician evaluation before any consideration.\n")
    lines.append("**Evidence:** 1 = in-vitro only | 2 = animal models | 3 = Phase I/II | 4 = Phase III adjacent\n")

    if state.off_label_hypotheses:
        lines += ["| Drug | Approved For | Shared Mechanism | Evidence | Source |",
                  "|------|-------------|-----------------|----------|--------|"]
        for h in state.off_label_hypotheses:
            flag = " ⚠️" if h.evidence_level < 2 else ""
            lines.append(
                f"| **{h.drug_name}** | {h.approved_indication} | "
                f"{h.shared_mechanism} | "
                f"{h.evidence_level} — {h.evidence_label}{flag} | "
                f"[PubMed]({h.citation}) |"
            )
        lines += ["", "> **Next step:** For drugs with evidence ≥ 3, ask: "
                      "*\"Has [drug] been considered given my [biomarker]?\"*"]
    else:
        lines.append("*No cross-indication hypotheses found for your biomarker profile.*")
    lines.append("")

    # ── Questions ────────────────────────────────────────────────────────────
    lines.append("## Questions to Ask Your Doctor\n")
    for t in valid_trials[:3]:
        lines.append(f'- Am I eligible for the trial "{t.title[:50]}..." ({t.nct_id})?')
    for h in [h for h in state.off_label_hypotheses if h.evidence_level >= 3]:
        genes = ", ".join(b.gene for b in state.biomarkers[:2])
        lines.append(f"- Has {h.drug_name} (approved for {h.approved_indication}) been considered for my {genes} profile?")
    if not valid_trials and not any(h.evidence_level >= 3 for h in state.off_label_hypotheses):
        lines += ["- What clinical trials are currently recruiting for my condition and biomarkers?",
                  "- Are there off-label treatments being studied for my specific mutation?"]
    lines.append("")

    # ── System Notes ─────────────────────────────────────────────────────────
    if state.validation_flags:
        lines.append("## System Notes\n")
        for flag in state.validation_flags:
            lines.append(f"- ⚠️ {flag}")
        lines.append("")

    lines.append(DISCLAIMER)
    state.final_report = "\n".join(lines)
    return state


print("✅ Agent 5 defined.")

---
## Cell 14 — Compile LangGraph Pipeline

In [ ]:
from langgraph.graph import StateGraph, END


def should_retry(state: PatientProfileState) -> str:
    critical = [f for f in state.validation_flags if "error" in f.lower()]
    if critical and state.retry_count < state.max_retries:
        state.retry_count += 1
        return "retry"
    return "done"


g = StateGraph(PatientProfileState)
g.add_node("intake",           agent_intake)
g.add_node("standard_care",    agent_standard_care)
g.add_node("clinical_trials",  agent_clinical_trials)
g.add_node("cross_indication", agent_cross_indication)
g.add_node("synthesizer",      agent_synthesizer)

g.set_entry_point("intake")
g.add_edge("intake",           "standard_care")
g.add_edge("standard_care",    "clinical_trials")
g.add_edge("clinical_trials",  "cross_indication")
g.add_edge("cross_indication", "synthesizer")
g.add_conditional_edges("synthesizer", should_retry, {"retry": "intake", "done": END})

pipeline = g.compile()
print("✅ Pipeline compiled and ready.")

---
## Cell 15 — ▶ Run the Pipeline

**Option A:** Type your condition description directly in `PATIENT_INPUT`.  
**Option B:** Upload a PDF medical report (set `USE_PDF = True`).

Then run this cell.

In [ ]:
# ─── Configure your input ────────────────────────────────────────────────────
USE_PDF = False   # Set True to upload a PDF instead

PATIENT_INPUT = """
I am a 54-year-old patient diagnosed with stage III non-small cell lung cancer (NSCLC).
My genomic report shows an EGFR exon 19 deletion mutation.
I am currently taking osimertinib (Tagrisso) as first-line treatment.
I have not had prior chemotherapy.
"""
# ─────────────────────────────────────────────────────────────────────────────

pdf_path = None
if USE_PDF:
    from google.colab import files as colab_files
    print("Select your PDF medical report:")
    uploaded_pdf = colab_files.upload()
    filename = list(uploaded_pdf.keys())[0]
    pdf_path = f"/tmp/{filename}"
    with open(pdf_path, "wb") as f:
        f.write(uploaded_pdf[filename])
    initial_state = PatientProfileState(raw_input=pdf_path, input_is_pdf=True)
else:
    initial_state = PatientProfileState(raw_input=PATIENT_INPUT.strip())

print("\n🔄 Running CDSS pipeline...")
print("  [1/5] Parsing patient profile...")
result = pipeline.invoke(initial_state)
print("  [2/5] Retrieving standard care guidelines...")
print("  [3/5] Searching clinical trials...")
print("  [4/5] Running cross-indication discovery...")
print("  [5/5] Compiling report...")
print("\n✅ Done.")

---
## Cell 16 — Display Report

In [ ]:
from IPython.display import Markdown, display

report = result["final_report"] if isinstance(result, dict) else result.final_report
display(Markdown(report))

---
## Cell 17 — Download Report

In [ ]:
from datetime import datetime
from google.colab import files as colab_files

output_filename = f"cdss_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(report)

colab_files.download(output_filename)
print(f"✅ Report downloaded as: {output_filename}")